# Problema 09: Amortiguador de Masa Sintonizado (TMD)

## Relacion 6 - Control de Vibraciones

**Objetivo:** Disenar un TMD (Tuned Mass Damper) optimo para reducir las vibraciones de una estructura sometida a excitacion armonica.

### Enunciado

Se tiene una estructura con las siguientes caracteristicas:
- Masa de la estructura: $m_s = 100$ toneladas
- Frecuencia natural: $\omega_s = 6.28$ rad/s (equivalente a $f_s = 1$ Hz)
- Amortiguamiento de la estructura: $\zeta_s = 2\%$

Se desea instalar un TMD con ratio de masa $\mu = 5\%$.

Se pide:
1. Determinar los parametros optimos del TMD segun Den Hartog
2. Comparar la respuesta con y sin TMD
3. Calcular la reduccion de la amplitud de vibracion

## Datos del Problema

**Estructura principal:**
| Parametro | Valor | Unidad |
|-----------|-------|--------|
| $m_s$ | 100,000 | kg |
| $\omega_s$ | 6.28 | rad/s |
| $\zeta_s$ | 2 | % |

**TMD:**
| Parametro | Valor | Unidad |
|-----------|-------|--------|
| $\mu$ | 5 | % |

In [ ]:
# Importacion de librerias
import numpy as np
import matplotlib.pyplot as plt

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# Parametros de la estructura principal
ms = 100e3     # kg - masa de la estructura
omega_s = 6.28  # rad/s - frecuencia natural
zeta_s = 0.02   # amortiguamiento de la estructura

# Propiedades derivadas
ks = ms * omega_s**2  # N/m - rigidez
cs = 2 * zeta_s * ms * omega_s  # Ns/m - amortiguamiento

print("PROPIEDADES DE LA ESTRUCTURA PRINCIPAL")
print("="*45)
print(f"Masa: ms = {ms/1000:.0f} ton")
print(f"Frecuencia: omega_s = {omega_s:.2f} rad/s (f = {omega_s/(2*np.pi):.2f} Hz)")
print(f"Amortiguamiento: zeta_s = {zeta_s*100:.0f}%")
print(f"\nPropiedades derivadas:")
print(f"  Rigidez: ks = {ks/1e6:.2f} MN/m")
print(f"  Coef. amortiguamiento: cs = {cs:.0f} Ns/m")

## 1. Parametros Optimos del TMD (Den Hartog)

Para un TMD optimo segun la formulacion de Den Hartog:

**Ratio de sintonizacion optimo:**
$$f_{opt} = \frac{\omega_d}{\omega_s} = \frac{1}{1 + \mu}$$

**Amortiguamiento optimo del TMD:**
$$\zeta_{d,opt} = \sqrt{\frac{3\mu}{8(1+\mu)}}$$

Donde $\mu = m_d/m_s$ es el ratio de masa.

In [ ]:
# Ratio de masa del TMD
mu = 0.05  # 5%

# Parametros optimos de Den Hartog
f_opt = 1 / (1 + mu)  # ratio de sintonizacion optimo
zeta_d_opt = np.sqrt(3*mu / (8*(1+mu)))  # amortiguamiento optimo

# Propiedades del TMD
md = mu * ms  # masa del TMD
omega_d = f_opt * omega_s  # frecuencia del TMD
kd = md * omega_d**2  # rigidez del TMD
cd = 2 * zeta_d_opt * md * omega_d  # amortiguamiento del TMD

print("PARAMETROS DEL TMD (Den Hartog)")
print("="*45)
print(f"Ratio de masa: mu = {mu*100:.1f}%")
print(f"\nParametros optimos:")
print(f"  f_opt = omega_d/omega_s = {f_opt:.4f}")
print(f"  zeta_d_opt = {zeta_d_opt*100:.2f}%")
print(f"\nPropiedades del TMD:")
print(f"  Masa: md = {md:.0f} kg ({md/1000:.1f} ton)")
print(f"  Frecuencia: omega_d = {omega_d:.2f} rad/s")
print(f"  Rigidez: kd = {kd:.0f} N/m ({kd/1e3:.1f} kN/m)")
print(f"  Coef. amortiguamiento: cd = {cd:.0f} Ns/m")

## 2. Funcion de Respuesta en Frecuencia (FRF)

### Sistema sin TMD (SDOF)

El factor de amplificacion dinamica para un SDOF es:
$$H(r) = \frac{1}{\sqrt{(1-r^2)^2 + (2\zeta_s r)^2}}$$

Donde $r = \omega/\omega_s$.

### Sistema con TMD (2DOF)

La FRF del sistema principal con TMD acoplado es mas compleja e involucra resolver el sistema de 2DOF.

In [ ]:
def FRF_sin_TMD(r, zeta_s):
    """
    Factor de amplificacion para SDOF sin TMD.
    
    Parametros:
        r: ratio de frecuencias omega/omega_s
        zeta_s: amortiguamiento de la estructura
    """
    return 1 / np.sqrt((1 - r**2)**2 + (2*zeta_s*r)**2)

def FRF_con_TMD(r, mu, f, zeta_s, zeta_d):
    """
    Factor de amplificacion para estructura con TMD.
    
    Parametros:
        r: ratio de frecuencias omega/omega_s
        mu: ratio de masa md/ms
        f: ratio de sintonizacion omega_d/omega_s
        zeta_s: amortiguamiento de la estructura
        zeta_d: amortiguamiento del TMD
    """
    H = np.zeros_like(r)
    
    for i, ri in enumerate(r):
        # Terminos complejos
        # Numerador: respuesta del TMD
        num = f**2 - ri**2 + 2j*zeta_d*f*ri
        
        # Denominador: sistema acoplado
        term1 = (1 - ri**2 + 2j*zeta_s*ri)
        term2 = (f**2 - ri**2 + 2j*zeta_d*f*ri)
        term3 = mu * f**2 * ri**2
        
        den = term1 * term2 - term3
        
        H[i] = np.abs(num / den)
    
    return H

# Rango de frecuencias
r = np.linspace(0.5, 1.5, 500)

# Calcular FRFs
H_sin = FRF_sin_TMD(r, zeta_s)
H_con = FRF_con_TMD(r, mu, f_opt, zeta_s, zeta_d_opt)

print("FRF CALCULADAS")
print("="*40)
print(f"Valor maximo sin TMD: H_max = {np.max(H_sin):.1f}")
print(f"Valor maximo con TMD: H_max = {np.max(H_con):.1f}")
print(f"Reduccion: {(1 - np.max(H_con)/np.max(H_sin))*100:.1f}%")

## 3. Visualizacion de la FRF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: FRF en escala logaritmica
ax1 = axes[0]
ax1.semilogy(r, H_sin, 'b-', linewidth=2, label='Sin TMD')
ax1.semilogy(r, H_con, 'r-', linewidth=2, label='Con TMD optimo')

# Linea de resonancia
ax1.axvline(x=1.0, color='gray', linestyle=':', linewidth=1)
ax1.text(1.02, 20, r'$\omega = \omega_s$', fontsize=10)

# Valores maximos
H_max_sin = np.max(H_sin)
H_max_con = np.max(H_con)
ax1.axhline(y=H_max_sin, color='blue', linestyle='--', linewidth=1, alpha=0.5)
ax1.axhline(y=H_max_con, color='red', linestyle='--', linewidth=1, alpha=0.5)

# Anotacion de reduccion
reduccion = (1 - H_max_con/H_max_sin) * 100
ax1.text(0.55, 15, f'Reduccion: {reduccion:.0f}%', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax1.set_xlabel(r'Frecuencia normalizada $\omega/\omega_s$', fontsize=12)
ax1.set_ylabel('Factor de amplificacion |H|', fontsize=12)
ax1.set_title('Funcion de Respuesta en Frecuencia', fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3, which='both')
ax1.set_xlim(0.5, 1.5)
ax1.set_ylim(0.1, 50)

# Panel 2: FRF en escala lineal
ax2 = axes[1]
ax2.plot(r, H_sin, 'b-', linewidth=2, label='Sin TMD')
ax2.plot(r, H_con, 'r-', linewidth=2, label='Con TMD optimo')
ax2.axvline(x=1.0, color='gray', linestyle=':', linewidth=1)

# Marcar los dos picos del sistema con TMD
peaks_idx = np.where((np.diff(np.sign(np.diff(H_con))) < 0))[0] + 1
for idx in peaks_idx:
    ax2.plot(r[idx], H_con[idx], 'ro', markersize=8)
    ax2.annotate(f'({r[idx]:.2f}, {H_con[idx]:.1f})', 
                 (r[idx], H_con[idx]), textcoords="offset points",
                 xytext=(5, 5), fontsize=9)

ax2.set_xlabel(r'Frecuencia normalizada $\omega/\omega_s$', fontsize=12)
ax2.set_ylabel('Factor de amplificacion |H|', fontsize=12)
ax2.set_title('FRF (escala lineal)', fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0.5, 1.5)

plt.tight_layout()
plt.show()

## 4. Efecto de la Variacion del Ratio de Masa

In [ ]:
# Estudio parametrico para diferentes ratios de masa
mus = [0.01, 0.02, 0.05, 0.10, 0.20]

fig, ax = plt.subplots(figsize=(10, 6))

# Sistema sin TMD
ax.semilogy(r, H_sin, 'k--', linewidth=2, label='Sin TMD')

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(mus)))

print("EFECTO DEL RATIO DE MASA")
print("="*50)
print(f"{'mu (%)':<10} {'f_opt':<10} {'zeta_d (%)':<12} {'H_max':<10} {'Reduccion (%)':<12}")
print("-"*50)

for i, mu_i in enumerate(mus):
    f_i = 1 / (1 + mu_i)
    zeta_d_i = np.sqrt(3*mu_i / (8*(1+mu_i)))
    H_i = FRF_con_TMD(r, mu_i, f_i, zeta_s, zeta_d_i)
    H_max_i = np.max(H_i)
    red_i = (1 - H_max_i/H_max_sin) * 100
    
    ax.semilogy(r, H_i, color=colors[i], linewidth=2, label=f'mu = {mu_i*100:.0f}%')
    print(f"{mu_i*100:<10.0f} {f_i:<10.4f} {zeta_d_i*100:<12.2f} {H_max_i:<10.1f} {red_i:<12.1f}")

ax.axvline(x=1.0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel(r'Frecuencia normalizada $\omega/\omega_s$', fontsize=12)
ax.set_ylabel('Factor de amplificacion |H|', fontsize=12)
ax.set_title('Efecto del ratio de masa del TMD', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0.5, 1.5)
ax.set_ylim(0.5, 30)

plt.tight_layout()
plt.show()

## 5. Sensibilidad a la Sintonizacion

In [ ]:
# Efecto de la desintonizacion
desviaciones = [-0.10, -0.05, 0, 0.05, 0.10]  # desviacion respecto a f_opt

fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(r, H_sin, 'k--', linewidth=2, label='Sin TMD')

colors = plt.cm.coolwarm(np.linspace(0, 1, len(desviaciones)))

print("EFECTO DE LA DESINTONIZACION")
print("="*50)
print(f"{'Desviacion (%)':<15} {'f':<10} {'H_max':<10} {'Perdida eficiencia (%)':<20}")
print("-"*50)

H_max_opt = np.max(FRF_con_TMD(r, mu, f_opt, zeta_s, zeta_d_opt))

for i, desv in enumerate(desviaciones):
    f_i = f_opt * (1 + desv)
    H_i = FRF_con_TMD(r, mu, f_i, zeta_s, zeta_d_opt)
    H_max_i = np.max(H_i)
    perdida = (H_max_i - H_max_opt) / (H_max_sin - H_max_opt) * 100
    
    label = f'f = {f_i:.3f}' + ('' if desv == 0 else f' ({desv*100:+.0f}%)')
    ls = '-' if desv == 0 else '--'
    ax.semilogy(r, H_i, color=colors[i], linewidth=2, linestyle=ls, label=label)
    
    print(f"{desv*100:<15.0f} {f_i:<10.4f} {H_max_i:<10.1f} {perdida:<20.1f}")

ax.axvline(x=1.0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel(r'Frecuencia normalizada $\omega/\omega_s$', fontsize=12)
ax.set_ylabel('Factor de amplificacion |H|', fontsize=12)
ax.set_title('Sensibilidad a la sintonizacion del TMD', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0.5, 1.5)
ax.set_ylim(0.5, 30)

plt.tight_layout()
plt.show()

## 6. Resumen de Diseno

In [ ]:
print("")
print("="*60)
print("      RESUMEN DEL DISENO DEL TMD")
print("="*60)
print("")
print("ESTRUCTURA PRINCIPAL:")
print(f"  - Masa: ms = {ms/1000:.0f} ton")
print(f"  - Frecuencia: fs = {omega_s/(2*np.pi):.2f} Hz")
print(f"  - Amortiguamiento: zeta_s = {zeta_s*100:.0f}%")
print("")
print("TMD OPTIMO (Den Hartog):")
print(f"  - Ratio de masa: mu = {mu*100:.1f}%")
print(f"  - Masa del TMD: md = {md:.0f} kg ({md/1000:.1f} ton)")
print(f"  - Frecuencia: fd = {omega_d/(2*np.pi):.3f} Hz")
print(f"  - Rigidez: kd = {kd/1000:.1f} kN/m")
print(f"  - Amortiguamiento: zeta_d = {zeta_d_opt*100:.2f}%")
print(f"  - Coef. amortiguamiento: cd = {cd:.0f} Ns/m")
print("")
print("RENDIMIENTO:")
print(f"  - H_max sin TMD: {H_max_sin:.1f}")
print(f"  - H_max con TMD: {np.max(H_con):.1f}")
print(f"  - Reduccion: {(1 - np.max(H_con)/H_max_sin)*100:.1f}%")
print("")
print("="*60)

## Interpretacion de Resultados

### Efecto del TMD
- El TMD divide el pico unico de resonancia en dos picos menores.
- Los dos nuevos picos corresponden a los modos del sistema acoplado (2DOF).
- La reduccion de la amplitud maxima depende principalmente del ratio de masa.

### Parametros optimos de Den Hartog
- El ratio de sintonizacion $f_{opt} = 1/(1+\mu)$ es ligeramente menor que 1.
- Esto "desintona" el TMD respecto a la estructura para igualar los dos picos.
- El amortiguamiento optimo crece con el ratio de masa.

### Consideraciones practicas
1. **Ratio de masa**: Tipicamente entre 1-10%. Mayor ratio implica mayor reduccion pero mayor peso y coste.
2. **Sintonizacion**: El TMD es sensible a la desintonizacion. Se recomienda margen de seguridad.
3. **Robustez**: Para estructuras con frecuencia variable, considerar TMD con amortiguamiento mayor.
4. **Espacio**: El TMD requiere espacio para el movimiento relativo (stroke).

### Aplicaciones tipicas
- Edificios altos sometidos a viento
- Pasarelas peatonales (vibraciones por uso)
- Torres de comunicaciones
- Chimeneas industriales